# JORINOVA NEXUS - Viral cytopathology / inclusion-body detector

Detects the **cellular footprints of viral infection** on histology / cytology / Tzanck smear:
**CMV** owl-eye, **HSV/VZV** multinucleation, **HPV** koilocytes, **molluscum** bodies,
**Negri** bodies (rabies). Viruses themselves are sub-microscopic. Each finding resolves to its
virus/disease via `backend/ai_services/virology_cyto_findings.json`.

> Screening aid - a pathologist confirms; molecular (PCR/IHC) proves the virus.

**Before you start:** Runtime -> **Change runtime type** -> **T4 GPU**.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo 'NO GPU - set Runtime > Change runtime type > T4 GPU, then rerun'

## 2. Install dependencies

In [ ]:
%pip -q install ultralytics roboflow
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| cuda', torch.cuda.is_available())

## 3. Mount Drive (checkpoints survive a runtime reset)

In [ ]:
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/nexus_virology_cyto', exist_ok=True)
    print('Drive ready: /content/drive/MyDrive/nexus_virology_cyto')
except Exception as e:
    print('Drive not mounted (optional):', e)

## 4. Get the training data (Roboflow)
Search **https://universe.roboflow.com/search?q=inclusion%20bodies** (also `cmv`, `koilocyte`, `tzanck`, `molluscum`),
open a **DETECTION** project, and paste its `(workspace, project)` into `CANDIDATES`.

In [ ]:
# -- Data: Roboflow -> DATA_YAML (prints the REAL error per candidate) --
# Add a DETECTION project from: https://universe.roboflow.com/search?q=inclusion%20bodies
#   (Download -> YOLOv8 -> "show download code" gives workspace() + project()).
import os, glob, yaml, shutil
from getpass import getpass
key = getpass('Roboflow Private API Key: ').strip(); assert key, 'EMPTY key -- paste the Private API Key.'
os.environ['ROBOFLOW_API_KEY'] = key
try:
    from roboflow import Roboflow
except ModuleNotFoundError:
    import subprocess, sys; subprocess.run([sys.executable,'-m','pip','install','-q','roboflow'], check=True)
    from roboflow import Roboflow
rf = Roboflow(api_key=key)

CANDIDATES = [
    # ('workspace-slug', 'project-slug'),   # <- add a project from the search URL above
]
assert CANDIDATES, 'Add a (workspace, project) from the search URL above.'
LOC = '/content/data/vcyto_rf'
if os.path.exists(LOC): shutil.rmtree(LOC)
ds = None
for w, p in CANDIDATES:
    try:
        proj = rf.workspace(w).project(p)
        vers = sorted([v.version for v in proj.versions()], reverse=True) or list(range(11, 0, -1))
        for v in vers:
            try:
                ds = proj.version(v).download('yolov8', location=LOC); print('OK', w+'/'+p, 'v'+str(v)); break
            except Exception as e: print('  v%s -> %r' % (v, str(e)[:90]))
        if ds: break
    except Exception as e:
        print('skip %s/%s -> %r' % (w, p, str(e)[:90]))
assert ds, 'None downloaded -- read errors above (401/403=key, 404=slug).'
DATA_YAML = os.path.join(ds.location, 'data.yaml')
print('DATA_YAML =', DATA_YAML, '| classes:', yaml.safe_load(open(DATA_YAML))['names'])
print('If class names differ from virology_cyto_findings.json keys, send me the list -> I add aka-aliases.')

## 5. Fine-tune the detector
Microscopy -> full augmentation (rotation + both flips). Inclusions are small -> `imgsz=1024`.

In [ ]:
from ultralytics import YOLO
import os
PROJECT = '/content/drive/MyDrive/nexus_virology_cyto/runs'
try: os.makedirs(PROJECT, exist_ok=True)
except Exception: PROJECT = '/content/runs/detect'
model = YOLO('yolov8s.pt')
model.train(data=DATA_YAML, epochs=200, imgsz=1024, batch=8,
            project=PROJECT, name='virology_cyto', pretrained=True, patience=40, plots=True,
            cos_lr=True, hsv_h=0.0, hsv_s=0.7, hsv_v=0.4, degrees=180, fliplr=0.5, flipud=0.5,
            mosaic=1.0, translate=0.1, scale=0.4)
m = model.val(); print(f'mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')
try:
    for i, c in zip(m.box.ap_class_index, m.box.maps): print(f'  {model.names.get(int(i), i)}: mAP50-95={c:.3f}')
except Exception as e: print('per-class skipped:', e)

## 6b. RESUME if a run disconnects (run INSTEAD of step 5)

In [ ]:
import os, glob
from ultralytics import YOLO
runs = sorted(glob.glob('/content/drive/MyDrive/nexus_virology_cyto/runs/**/weights/last.pt', recursive=True), key=os.path.getmtime)
loose = [p for p in sorted(glob.glob('/content/drive/MyDrive/nexus_virology_cyto/**/*.pt', recursive=True) + glob.glob('/content/*.pt'), key=os.path.getmtime)
         if 'yolov8' not in os.path.basename(p).lower()]
assert runs or loose, 'No checkpoint yet -- run step 5 first.'
ckpt = (runs or loose)[-1]; print('checkpoint:', ckpt)
model = None
try:
    if ckpt in runs: model = YOLO(ckpt); model.train(resume=True)
    else: raise RuntimeError('loose')
except Exception as e:
    print('continue-from-weights:', str(e)[:80]); model = None
if model is None:
    assert 'DATA_YAML' in globals(), 'Re-run step 4 first so DATA_YAML is set.'
    model = YOLO(ckpt); model.train(data=DATA_YAML, epochs=200, imgsz=1024, batch=8,
        project='/content/drive/MyDrive/nexus_virology_cyto/runs', name='virology_cyto_cont', pretrained=True, patience=40,
        cos_lr=True, hsv_h=0.0, hsv_s=0.7, hsv_v=0.4, degrees=180, fliplr=0.5, flipud=0.5)
m = model.val(); print(f'mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')

## 7. Export the model (Drive backup + download)

In [ ]:
import shutil, glob, os
DRIVE = '/content/drive/MyDrive/nexus_virology_cyto'
best = sorted([c for c in glob.glob('/content/**/weights/best.pt', recursive=True)
               + glob.glob(DRIVE + '/**/weights/best.pt', recursive=True) if os.path.isfile(c)], key=os.path.getmtime)[-1]
print('using weights:', best)
from ultralytics import YOLO; print('classes:', YOLO(best).model.names)
try: os.makedirs(DRIVE, exist_ok=True); shutil.copy(best, DRIVE + '/virology_cyto.pt'); print('Drive ->', DRIVE + '/virology_cyto.pt')
except Exception as e: print('drive copy skipped:', e)
from google.colab import files; files.download(best)

## 8. Put it in the app
1. Rename the download to **`virology_cyto.pt`** at **`backend/models/virology_cyto/virology_cyto.pt`**, commit, push.
2. Auto-loads for `image_type` in {`viral_cytopathology`, `inclusion_bodies`, `virology_cyto`, `viral_inclusion`}.
3. Detected inclusion bodies -> virus/disease via `virology_cyto_findings.json` (CMV owl-eye -> CMV, koilocyte -> HPV, Negri -> rabies, ...).
4. On Render, build with `INSTALL_ML=1` (>=2 GB); else Claude vision reads the slide.